# Network Anomaly Detection

## Random Forests
A Random Forest trains many decision trees on random subsets of your data, then combines their outputs — majority vote for classification, average for regression — so the crowd's wisdom beats any single tree, cutting overfitting while staying robust across complex, high-dimensional datasets.

A trained model learns what "normal" looks like, then flags anything that doesn't fit as a potential anomaly — handy for catching things like suspicious network traffic.

## Downloading the NSL-KDD Dataset
NSL-KDD is a cleaned-up benchmark dataset of labeled normal and attack network traffic, commonly used to train and test intrusion detection models — supporting both binary (normal vs. attack) and multi-class (attack type) classification. We'll be using a modified version of it.

In [ ]:
import requests, zipfile, io

# URL for the NSL-KDD dataset
url = "https://academy.hackthebox.com/storage/modules/292/KDD_dataset.zip"

# Download the zip file and extract its contents
response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))
z.extractall('.')  # Extracts to the current directory

## Loading the Dataset
We begin by importing all necessary libraries.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt # visualization: confusion matrix & feature importance plots

### Defining Column Names and File Path
Map column names to the NSL-KDD features and set the file path for pandas to load the dataset.

In [ ]:
# Set the file path to the dataset
file_path = r'KDD+.txt'

# Define the column names corresponding to the NSL-KDD dataset
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 
    'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack', 'level'
]

### Reading the Dataset into a DataFrame
With the file path and column names defined, we load the data into a pandas DataFrame. This provides a structured, tabular representation of the dataset, making it easier to inspect, preprocess, and visualize.

The DataFrame is ready for further inspection, cleaning, and preprocessing steps. Before proceeding, we can briefly examine the dataset’s structure, check for missing values, and confirm that all features align with their intended data types.

In [ ]:
# Read the combined NSL-KDD dataset into a DataFrame
df = pd.read_csv(file_path, names=columns)

print(df.head())

# Preprocessing and Splitting the Dataset

## Preprocessing the Dataset
Prepare the NSL-KDD dataset for training by building two classification targets (binary: normal vs. attack, multi-class: attack type), encoding categorical variables into numeric form, and selecting the relevant numeric features — producing a clean, model-ready dataset.

### Creating a Binary Classification Target
Create a binary `attack_flag` column that labels each row as `0` (normal) or `1` (attack), reducing the problem to a simple normal-vs-attack classification before tackling more granular multi-class detection.

In [ ]:
# Binary classification target
# Maps normal traffic to 0 and any type of attack to 1
df['attack_flag'] = df['attack'].apply(lambda a: 0 if a == 'normal' else 1)

### Creating the Multi-Class Classification Target
Extend binary labels into a 5-class target by mapping each attack name to its category — normal (0), DoS (1), Probe (2), Privilege Escalation (3), or Access (4) — giving the model enough granularity to identify not just *that* an attack occurred, but *what kind*.

In [ ]:
# Multi-class classification target categories
dos_attacks = ['apache2', 'back', 'land', 'neptune', 'mailbomb', 'pod', 
               'processtable', 'smurf', 'teardrop', 'udpstorm', 'worm']
probe_attacks = ['ipsweep', 'mscan', 'nmap', 'portsweep', 'saint', 'satan']
privilege_attacks = ['buffer_overflow', 'loadmdoule', 'perl', 'ps', 
                     'rootkit', 'sqlattack', 'xterm']
access_attacks = ['ftp_write', 'guess_passwd', 'http_tunnel', 'imap', 
                  'multihop', 'named', 'phf', 'sendmail', 'snmpgetattack', 
                  'snmpguess', 'spy', 'warezclient', 'warezmaster', 
                  'xclock', 'xsnoop']

def map_attack(attack):
    if attack in dos_attacks:
        return 1
    elif attack in probe_attacks:
        return 2
    elif attack in privilege_attacks:
        return 3
    elif attack in access_attacks:
        return 4
    else:
        return 0

# Assign multi-class category to each row
df['attack_map'] = df['attack'].apply(map_attack)

### Encoding Categorical Variables
One-hot encode `protocol_type` and `service` into binary indicator columns using `pd.get_dummies`, converting categorical network attributes into numeric form without implying any ordinal relationship between categories.

If you encode `tcp=1, udp=2, icmp=3`, the model might learn "icmp is 3x tcp" or weight higher numbers more — total nonsense for protocols. One-hot gives you:

| tcp | udp | icmp |
|-----|-----|------|
| 1   | 0   | 0    |
| 0   | 1   | 0    |
| 0   | 0   | 1    |

Each protocol is just present or absent — no false relationships.

In [ ]:
# Encoding categorical variables
features_to_encode = ['protocol_type', 'service']
encoded = pd.get_dummies(df[features_to_encode])

### Selecting Numeric Features
Select the numeric features — spanning raw traffic metrics like `src_bytes` and `dst_bytes` through derived statistical rates like `serror_rate` — that give the model both volume signals and behavioral patterns for distinguishing normal from attack traffic.

In [14]:
# Numeric features that capture various statistical properties of the traffic
numeric_features = [
    'duration', 'src_bytes', 'dst_bytes', 'wrong_fragment', 'urgent', 'hot', 
    'num_failed_logins', 'num_compromised', 'root_shell', 'su_attempted', 
    'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 
    'num_outbound_cmds', 'count', 'srv_count', 'serror_rate', 
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 
    'dst_host_srv_rerror_rate'
]

### Preparing the Dataset
Concatenate the one-hot encoded categoricals with the numeric features into a single `train_set` DataFrame, and store the multi-class labels as `multi_y` — the dataset is now fully prepared for train/test splitting and model training.

Three types across the full pipeline:

- **numeric** — raw and derived traffic stats (int/float)
- **bool** — one-hot encoded protocol/service columns
- **int** — `attack_flag` (0/1) and `attack_map` (0-4) labels
- 
`multi_y` is the target/label, not a feature — you never train on what you're trying to predict. It stays separate so you can pass train_set (features) and multi_y (labels) independently into the model.

In [15]:
# Combine encoded categorical variables and numeric features
train_set = encoded.join(df[numeric_features])

# Multi-class target variable
multi_y = df['attack_map']

## Splitting the Dataset
Apply the train/validation/test split to the prepared NSL-KDD data, ensuring the random forest is trained, tuned, and evaluated on strictly separate subsets to give an honest measure of real-world performance.

### Splitting Data into Training and Test Sets
We use train_test_split to allocate a portion of the data for testing, ensuring that our final evaluations occur on unseen data.

In [18]:
# Split data into training and test sets for multi-class classification
train_X, test_X, train_y, test_y = train_test_split(train_set, multi_y, test_size=0.2, random_state=1337)

### Creating a Validation Set from the Training Data
We further split the training data to create a validation set. This supports model tuning and hyperparameter optimization without contaminating the final test data.

Four sets total after both splits:

- **`multi_train_X/y`** — trains the model
- **`multi_val_X/y`** — tunes hyperparameters during development
- **`test_X/y`** — final honest evaluation, touched only once at the end

`train_X/y` is now just an intermediate that got split further into train and val.

X is features (input), y is labels (target). Always pairs because every row in X needs its corresponding label in y to be meaningful.

In [19]:
# Further split the training set into separate training and validation sets
multi_train_X, multi_val_X, multi_train_y, multi_val_y = train_test_split(train_X, train_y, test_size=0.3, random_state=1337)

### Final Split Variables
After splitting, we have:

- train_X, train_y: Core training set
- test_X, test_y: Reserved for the final performance evaluation
- multi_train_X, multi_train_y: Training subset for fitting the model
- multi_val_X, multi_val_y: Validation subset for hyperparameter tuning